In [1]:
# Install packages from requirements.txt
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 48.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.7/949.7 kB 37.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 47.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 60.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 49.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 23.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 51.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 51.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.5/803.5 kB 36.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 48.9 MB/s  0:00:006m0:0

In [2]:
# Auto-reload modules when they change
%load_ext autoreload
%autoreload 2

In [19]:
# Import all functions from utils
from utils import *

# Verify get_llm is imported
print("Available functions:", [name for name in dir() if not name.startswith('_')])

Available functions: ['AutoModelForCausalLM', 'AutoTokenizer', 'BertForSequenceClassification', 'BertTokenizer', 'ChatGroq', 'ConnectionError', 'Elasticsearch', 'ElasticsearchStore', 'F', 'In', 'Out', 'SentenceTransformer', 'advanced_query_routing', 'advanced_query_transformation', 'advanced_rag_pipeline', 'answer', 'chromadb', 'client', 'collection', 'collection_name', 'config', 'contents', 'df', 'documents', 'es', 'es_api_key', 'es_host', 'exit', 'faiss', 'fusion_retrieval', 'generate_answer', 'get_ipython', 'get_llm', 'hf_api_token', 'ids', 'index_name', 'json', 'load_config', 'login', 'movies_df', 'np', 'open', 'os', 'pd', 'pipeline', 'query', 'quit', 'rerank_documents', 'rerank_model', 'rerank_tokenizer', 'response', 'select_and_compress_context', 'sentence_model', 'summarizer']


In [20]:
import faiss
import numpy as np
from elasticsearch import Elasticsearch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
import os
import json

In [2]:
# Config is already loaded in utils.py, access it here
# print(f"GROQ_API_KEY: {config.get('GROQ_API_KEY')[:20]}...")
# print(f"Config loaded successfully!")


In [21]:
from huggingface_hub import login

# Your Hugging Face API token (You can find it in your Hugging Face account settings)
hf_api_token = config.get('HUGGING_FACE_API_KEY')


# Log in to Hugging Face
login(token=hf_api_token)

In [22]:
# Load the pre-trained models and tokenizers for text generation, sentence embedding,
# and reranking.

# Load the SentenceTransformer model for encoding queries and documents
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  # Small, fast model for embeddings

# Load the tokenizer for the reranking model
rerank_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load the BERT model for sequence classification (used for reranking)
rerank_model = BertForSequenceClassification.from_pretrained('bert-base-uncased')

# Load summarization model
summarizer = pipeline("summarization")

print("All models loaded successfully!")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
No model was supplied, defaulted to sshleifer/distilbart-cnn-12-6 and revision a4f8f3e (https://huggingface.co/sshleifer/distilbart-cnn-12-6).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu
Device set to use cpu


All models loaded successfully!


In [23]:
# Test the LLM function
response = get_llm().invoke("Hi")
print(response.content)


It's nice to meet you. Is there something I can help you with, or would you like to chat?


In [24]:
get_llm().invoke("Hi")

AIMessage(content="It's nice to meet you. Is there something I can help you with, or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 11, 'total_tokens': 34, 'completion_time': 0.020656669, 'prompt_time': 0.235607725, 'queue_time': 0.115735562, 'total_time': 0.256264394}, 'model_name': 'meta-llama/llama-4-maverick-17b-128e-instruct', 'system_fingerprint': 'fp_9b0c2006ef', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--b0164712-8e21-4812-8fac-800f05200b17-0', usage_metadata={'input_tokens': 11, 'output_tokens': 23, 'total_tokens': 34})

In [25]:
import chromadb

# Initialize ChromaDB client and create collection
client = chromadb.Client()

# Define the collection name
collection_name = "movies"

try:
    # Attempt to create the collection in ChromaDB
    collection = client.create_collection(name=collection_name)
    print(f"Collection '{collection_name}' created successfully.")

    # Define the documents to be inserted into the collection
    documents = [
        {"id": "1", "content": "The Shawshank Redemption is a great movie to watch on a rainy day."},
        {"id": "2", "content": "Forrest Gump is an uplifting film perfect for a rainy afternoon."}
    ]

    # Extract the IDs and content for insertion
    ids = [doc["id"] for doc in documents]
    contents = [doc["content"] for doc in documents]

    # Insert documents into the collection
    collection.add(ids=ids, documents=contents)
    print("Documents inserted successfully.")

except Exception as e:
    print(f"Collection '{collection_name}' already exists. No need to create it again.")
    # Optionally, you could fetch the existing collection here
    collection = client.get_collection(name=collection_name)

except Exception as e:
    print(f"An error occurred: {e}")


Collection 'movies' already exists. No need to create it again.


In [26]:
def advanced_rag_pipeline(query, collection, documents, es, index_name='movies'):
    """
    The main pipeline function for the Advanced Retrieval-Augmented Generation (RAG) system.
    It processes the query, retrieves relevant documents, reranks them, selects and compresses
    the context, and finally generates an answer.

    Args:
        query (str): The user's input query.
        collection: ChromaDB collection
        documents: List of documents
        es: Elasticsearch client
        index_name (str): Elasticsearch index name

    Returns:
        str: The final generated answer.
    """
    # Transform and route query
    transformed_query = advanced_query_transformation(query)
    retrieval_method = advanced_query_routing(transformed_query)

    # Retrieve documents using fusion retrieval
    retrieved_documents = fusion_retrieval(transformed_query, collection, documents, es, index_name)

    # Rerank documents based on relevance
    ranked_documents = rerank_documents(query, retrieved_documents, rerank_tokenizer, rerank_model)

    # Select and compress context for answer generation
    context = select_and_compress_context(ranked_documents, summarizer)

    # Get LLM instance
    llm = get_llm()
    
    # Generate final answer based on the context
    final_answer = generate_answer(query, context, llm)
    return final_answer


In [27]:
from elasticsearch import Elasticsearch

# Load Elasticsearch credentials from config
es_host = config.get('ELASTICSEARCH_HOST')
es_api_key = config.get('ELASTICSEARCH_PASSWORD')  # This is actually the API key

# Initialize Elasticsearch client with API key
es = Elasticsearch(
    es_host,
    api_key=es_api_key
)

# Test connection
es.ping()

True

In [28]:
# Example query
query = "What are some good movies to watch on a rainy day?"

# Define index name
index_name = 'movies'

# Run the query through the Advanced RAG Pipeline
answer = advanced_rag_pipeline(query, collection, documents, es, index_name)

# Output the generated answer
print(answer)


Based on the context, it appears that you're looking for movie suggestions that are uplifting, great, or otherwise engaging to watch on a rainy day. Given that Forrest Gump is described as an uplifting film and The Shawshank Redemption is considered a great movie, I'll suggest a list of films that share similar qualities or are otherwise suitable for a cozy rainy day.

Here are some movie suggestions:

1. **Uplifting Films:**
   - **Forrest Gump (1994)** - As mentioned, it's an uplifting film that follows the life of Forrest Gump, played by Tom Hanks, through significant historical events.
   - **The Pursuit of Happyness (2006)** - A true story about perseverance and chasing your dreams, starring Will Smith.
   - **Hidden Figures (2016)** - An inspiring story about three African-American women who worked as mathematicians and engineers at NASA during the early years of the space program.

2. **Great Dramas:**
   - **The Shawshank Redemption (1994)** - Described as a great movie, it's a

In [13]:
# Setup Kaggle credentials
# Option 1: Upload kaggle.json manually, then run:
!mkdir -p ~/.kaggle
# If you uploaded kaggle.json to the workspace root:
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [14]:
# Download the Movies Dataset from Kaggle
!kaggle datasets download -d rounakbanik/the-movies-dataset

# Unzip the dataset
!unzip -o the-movies-dataset.zip

# List the files
!ls -lh *.csv

print("\n✅ Dataset downloaded and extracted!")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Dataset URL: https://www.kaggle.com/datasets/rounakbanik/the-movies-dataset
License(s): CC0-1.0
 58%|███████████████████████                 | 131M/228M [00:00<00:00, 1.37GB/s]
100%|████████████████████████████████████████| 228M/228M [00:00<00:00, 1.37GB/s]
Archive:  the-movies-dataset.zip
  inflating: credits.csv             

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



  inflating: keywords.csv            
  inflating: links.csv               
  inflating: links_small.csv         
  inflating: movies_metadata.csv     
  inflating: ratings.csv             
  inflating: ratings_small.csv       
-rw-r--r-- 1 codespace codespace 182M Sep 21  2019 credits.csv
-rw-r--r-- 1 codespace codespace 6.0M Sep 21  2019 keywords.csv
-rw-r--r-- 1 codespace codespace 966K Sep 21  2019 links.csv
-rw-r--r-- 1 codespace codespace 180K Sep 21  2019 links_small.csv
-rw-r--r-- 1 codespace codespace  33M Sep 21  2019 movies_metadata.csv
-rw-r--r-- 1 codespace codespace 677M Sep 21  2019 ratings.csv
-rw-r--r-- 1 codespace codespace 2.4M Sep 21  2019 ratings_small.csv

✅ Dataset downloaded and extracted!


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [29]:
import pandas as pd

# Load the movies dataset
df = pd.read_csv('movies_metadata.csv', low_memory=False)

# Select top 5000 rows and relevant columns
movies_df = df.loc[:4999, ['original_title', 'overview']].copy()

# Drop rows with missing overviews
movies_df = movies_df.dropna(subset=['overview'])

# Reset index
movies_df = movies_df.reset_index(drop=True)

print(f"Loaded {len(movies_df)} movies with overviews")
movies_df.head()

Loaded 4979 movies with overviews


,original_title,overview
0,Toy Story,"Led by Woody, Andy's toys live happily in his ..."
1,Jumanji,When siblings Judy and Peter discover an encha...
2,Grumpier Old Men,A family wedding reignites the ancient feud be...
3,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom..."
4,Father of the Bride Part II,Just when George Banks has recovered from his ...


## Chunk Movie Overviews

Split the movie overviews into smaller chunks for better retrieval

In [9]:
# Download required NLTK data
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')
print("✅ NLTK data downloaded!")

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /home/codespace/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


✅ NLTK data downloaded!


# Load and Prepare Movies Dataset

We'll load the top 5000 movies from movies_metadata.csv and prepare them for RAG

In [17]:
# Test query
query = "What are some good action movies with great visual effects?"

print(f"Query: {query}\n")
print("Running Advanced RAG Pipeline...\n")

# Run the pipeline
answer = advanced_rag_pipeline(query, collection, documents, es, index_name)

print("="*80)
print("ANSWER:")
print("="*80)
print(answer)

Query: What are some good action movies with great visual effects?

Running Advanced RAG Pipeline...

ANSWER:
Based on the context provided, it seems like we're discussing highly acclaimed and emotionally resonant films. "Forrest Gump" is known for its uplifting and inspiring storyline, while "The Shawshank Redemption" is often cited as one of the greatest films of all time due to its powerful narrative and emotional depth. Although neither film is primarily classified as an action movie, they both have elements that could be used to infer preferences for certain types of cinematic experiences.

To suggest action movies with great visual effects, let's consider a few categories and examples:

1. **Science Fiction/Action Films**: These often blend action with impressive visual effects.
   - "Avatar" (2009) - Known for its groundbreaking CGI and 3D technology.
   - "The Matrix" (1999) - Pioneered the use of "bullet time" and innovative action sequences.
   - "Interstellar" (2014) - Featu

# Test the Advanced RAG Pipeline

Now let's test the complete pipeline with real movie data

In [18]:
# Define index name
index_name = 'movies'

# Delete index if exists
if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
    print(f"Deleted existing index '{index_name}'")

# Create new index with mapping
es.indices.create(
    index=index_name,
    mappings={
        "properties": {
            "content": {"type": "text"},
            "title": {"type": "text"}
        }
    }
)
print(f"Created index '{index_name}'")

# Insert movie chunks into Elasticsearch
print("Inserting data into Elasticsearch...")
for idx, row in chunked_df.iterrows():
    es.index(
        index=index_name,
        id=str(idx),
        document={
            "content": str(row['chunks']),
            "title": str(row['original_title'])
        }
    )

print(f"✅ Successfully stored {len(chunked_df)} movie chunks in Elasticsearch!")

Deleted existing index 'movies'
Created index 'movies'
Inserting data into Elasticsearch...


NameError: name 'chunked_df' is not defined

## Store in Elasticsearch

Create index and insert movie chunks into Elasticsearch

In [ ]:
from elasticsearch import Elasticsearch

# Load Elasticsearch credentials from config
es_host = config.get('ELASTICSEARCH_HOST')
es_api_key = config.get('ELASTICSEARCH_PASSWORD')

# Initialize Elasticsearch client
es = Elasticsearch(es_host, api_key=es_api_key)

# Test connection
if es.ping():
    print("✅ Connected to Elasticsearch")
else:
    print("❌ Elasticsearch connection failed")

## Setup Elasticsearch Connection

Connect to Elasticsearch and verify the connection

In [ ]:
# Initialize ChromaDB client
client = chromadb.Client()

# Delete collection if it exists
try:
    client.delete_collection(name="movies")
    print("Deleted existing 'movies' collection")
except:
    pass

# Create new collection
collection = client.create_collection(name="movies")

# Insert data into ChromaDB
print("Inserting data into ChromaDB...")
for idx, row in chunked_df.iterrows():
    collection.add(
        ids=[str(idx)],
        embeddings=[row['embeddings']],
        metadatas=[{
            'original_title': str(row['original_title']),
            'chunk': str(row['chunks'])
        }]
    )

# Store documents list for RAG pipeline
documents = chunked_df['chunks'].tolist()

print(f"✅ Successfully stored {len(chunked_df)} movie chunks in ChromaDB!")

## Store in ChromaDB

Insert the movie chunks and embeddings into ChromaDB

In [ ]:
def encode_chunk(chunk):
    """Encode a text chunk into embeddings."""
    if not isinstance(chunk, str) or chunk.strip() == "":
        return None
    return sentence_model.encode(chunk).tolist()

print("Creating embeddings... This may take a few minutes.")
chunked_df['embeddings'] = chunked_df['chunks'].apply(encode_chunk)

# Drop rows with None embeddings
chunked_df.dropna(subset=['embeddings'], inplace=True)

print(f"✅ Created embeddings for {len(chunked_df)} chunks")

## Create Embeddings

Generate vector embeddings for each chunk using SentenceTransformer

In [30]:
from langchain.text_splitter import NLTKTextSplitter

# Initialize text splitter
text_splitter = NLTKTextSplitter(chunk_size=500)

def split_overview(overview):
    """Split movie overview into chunks."""
    if pd.isna(overview) or not isinstance(overview, str):
        return []
    return text_splitter.split_text(str(overview))

# Create chunks for each movie
movies_df['chunks'] = movies_df['overview'].apply(split_overview)

# Explode to get one row per chunk
chunked_df = movies_df.explode('chunks').reset_index(drop=True)

# Remove empty chunks
chunked_df = chunked_df[chunked_df['chunks'].str.strip() != '']
chunked_df = chunked_df.reset_index(drop=True)

print(f"Created {len(chunked_df)} chunks from {len(movies_df)} movies")
chunked_df.head(10)

ModuleNotFoundError: No module named 'langchain'

In [31]:
!pip install langchain

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
    Uninstalling langchain-core-0.3.79:
      Successfully uninstalled langchain-core-0.3.79
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain]/8 [langchain]core]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 0.3.8 requires langchain-core<1.0.0,>=0.3.75, but you have langchain-core 1.1.0 which is incompatible.
langchain-huggingface 0.3.1 requires langchain-core<1.0.0,>=0.3.70, but you have langchain-core 1.1.0 which is incompatible.
langchain-elasticsearch 0.4.0 requires langchain-core<0.4.0,>=0.3.0, but you have langchain-core 1.1.0 which is incompatible.
   